## *Recolha e Pré-processamento*

In [3]:
from Bio import Entrez
import pandas as pd

Entrez.email = "conhecimentolinguagem@gmail.com"

term = '("disease"[MeSH Terms]) AND ("symptom"[Title/Abstract] OR "treatment"[Title/Abstract]) AND ("2021"[Date - Publication] : "2025"[Date - Publication])'

# Pesquisa
handle = Entrez.esearch(db="pubmed", term=term, retmax=1)  # podes aumentar retmax
record = Entrez.read(handle)
ids = record["IdList"]

articles = []

for pmid in ids:
    fetch = Entrez.efetch(db="pubmed", id=pmid, rettype="xml")
    data = Entrez.read(fetch)
    
    article_data = data['PubmedArticle'][0]['MedlineCitation']['Article']
    
    title = article_data.get('ArticleTitle', '')
    
    # Abstract pode ter várias partes (<AbstractText> pode ser lista)
    abstract_text = ''
    if 'Abstract' in article_data and 'AbstractText' in article_data['Abstract']:
        abstract_parts = article_data['Abstract']['AbstractText']
        if isinstance(abstract_parts, list):
            # Junta todas as partes
            abstract_text = ' '.join([str(part) for part in abstract_parts])
        else:
            abstract_text = str(abstract_parts)
    
    articles.append({
        'pmid': pmid,
        'title': title,
        'abstract': abstract_text
    })

df = pd.DataFrame(articles)
df.to_csv("articles.csv", index=False)
print(df)

       pmid                                              title  \
0  41213757  Successful management of parvovirus B19-associ...   

                                            abstract  
0  Maternal mirror syndrome (MMS), or Ballantyne ...  


## *Extração de Entidades*

- Criar um ambiente virtual novo
- pip install scapy==3.7.4
- pip install scispacy==0.5.1
- Download de "en_ner_bc5cdr_md" em https://allenai.github.io/scispacy/
- pip install "location"


#### *Transformers*

In [79]:
import spacy
import re
import csv
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import pandas as pd

# -------------------------------
# --- Modelo NER ---
# -------------------------------
model_names = ["d4data/biomedical-ner-all"]
ner_pipelines = []
for name in model_names:
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForTokenClassification.from_pretrained(name)
    ner_pipelines.append(pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple"))

# -------------------------------
# --- SciSpaCy ---
# -------------------------------
nlp_spacy = spacy.load("en_ner_bc5cdr_md")  # Disease e Chemical

# -------------------------------
# --- DrugBank ---
# -------------------------------
drugbank_list = []
try:
    with open("drugbank_dataset.csv", newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            drugbank_list.append(row['Common name'].strip().lower())
            if row.get('Synonyms'):
                for syn in row['Synonyms'].split('|'):
                    syn = syn.strip().lower()
                    if syn:
                        drugbank_list.append(syn)
    drugbank_list = list(set(drugbank_list))
    print(f"Carregados {len(drugbank_list)} termos do DrugBank.")
except FileNotFoundError:
    print("Aviso: 'drugbank_dataset.csv' não encontrado. A lista de tratamentos pode estar incompleta.")
    drugbank_list = []


# -------------------------------
# --- Lista de sintomas comuns ---
# -------------------------------
sintomas_comuns = {
    "fever", "cough", "fatigue", "headache", "nausea", "vomiting",
    "shortness of breath", "pain", "swelling", "stiffness", "dizziness",
    "chills", "sore throat", "diarrhea", "rash", "itching"
}

# -------------------------------
# --- Funções auxiliares ---
# -------------------------------
def unir_substrings(entidades_set):
    """Remove entidades redundantes ou sobrepostas, priorizando a mais longa."""
    entidades = sorted(entidades_set, key=lambda x: -len(x))
    final = set()
    for e in entidades:
        # Adiciona a entidade 'e' se nenhuma entidade 'f' já em 'final' for uma superstring de 'e'
        # E se 'e' não for uma substring de nenhuma entidade 'f' já em 'final'
        # (Esta lógica simplificada prioriza a entidade mais longa que aparece primeiro)
        if not any(e in f for f in final):
            # Remove quaisquer substrings de 'e' que já possam estar em 'final'
            final = {f for f in final if f not in e}
            final.add(e)
    return final

def fundir_entidades_adjacentes(entidades_set, texto_lower):
    """
    Tenta fundir entidades na lista se a sua combinação existir no texto.
    Ex: Se "fetal" e "hydrops" estiverem no set, e "fetal hydrops" estiver no texto,
    substitui os dois por "fetal hydrops".
    """
    # Usamos um loop 'while' para permitir fusões recursivas
    # (ex: "severe" + "fetal" -> "severe fetal", depois "severe fetal" + "hydrops" -> "severe fetal hydrops")
    while True:
        houve_fusao = False
        entidades_list = sorted(list(entidades_set), key=len, reverse=True)
        novas_entidades = set()
        entidades_a_remover = set()

        # Copia o set para poder iterar com segurança
        entidades_a_verificar = set(entidades_set)

        for e1 in entidades_a_verificar:
            for e2 in entidades_a_verificar:
                if e1 == e2:
                    continue
                
                # Evita re-processar entidades já marcadas para remoção
                if e1 in entidades_a_remover or e2 in entidades_a_remover:
                    continue

                # Tenta combinar "e1 e2"
                combined_phrase = f"{e1} {e2}"

                # Se a frase combinada existir no texto...
                if combined_phrase in texto_lower:
                    # Verifica se esta combinação já não é uma substring de algo maior
                    if not any(combined_phrase in f and combined_phrase != f for f in entidades_set):
                        print(f"    → Fusão: '{e1}' + '{e2}' -> '{combined_phrase}'")
                        novas_entidades.add(combined_phrase)
                        entidades_a_remover.add(e1)
                        entidades_a_remover.add(e2)
                        houve_fusao = True

        # Aplica as mudanças da passagem
        if houve_fusao:
            entidades_set.update(novas_entidades)
            entidades_set.difference_update(entidades_a_remover)
        else:
            # Se não houve fusões nesta passagem, o processo está completo
            break

    return entidades_set

# -------------------------------
# --- Função principal (Versão Otimizada) ---
# -------------------------------
def extrair_entidades(texto):
    texto_lower = texto.lower()
    entity_votes = defaultdict(list)  # entidade -> lista de labels

    # --- AJUSTE 1: FILTRO DE CONFIANÇA ---
    # Ignora deteções dos transformers com score abaixo de 50%
    # Isto resolve o problema do "premature" (score 0.31)
    CONFIDENCE_THRESHOLD = 0.5 

    print("\n[DEBUG] --- Transformers Ensemble ---")
    for ner in ner_pipelines:
        print(f"\nModelo: {ner.model.name_or_path}")
        resultados = ner(texto)
        for r in resultados:
            token = r["word"].lstrip("#").lower().strip()
            token = re.sub(r'[^\w\s-]', '', token).strip()
            token = re.sub(r'\s+', ' ', token).strip()
            score = r['score']

            # APLICA O FILTRO DE CONFIANÇA
            if token and score >= CONFIDENCE_THRESHOLD:
                entity_votes[token].append(r['entity_group'])
                print(f"  - {token}: {r['entity_group']} ({score:.2f})")
            elif token:
                # Imprime o que foi rejeitado
                print(f"  - {token}: {r['entity_group']} ({score:.2f}) -- REJEITADO (Score baixo)")


    print("\n[DEBUG] --- SciSpaCy ---")
    doc = nlp_spacy(texto)
    for ent in doc.ents:
        entidade = texto[ent.start_char:ent.end_char].lower().strip()
        entidade = re.sub(r'[^\w\s-]', '', entidade).strip()
        entidade = re.sub(r'\s+', ' ', entidade).strip()

        if not entidade:
            continue

        if ent.label_ == "DISEASE":
            entity_votes[entidade].append("DISEASE")
            print(f"  - {entidade} [DISEASE] adicionada pelo SciSpaCy")
        elif ent.label_ == "CHEMICAL":
            entity_votes[entidade].append("CHEMICAL")
            print(f"  - {entidade} [CHEMICAL] adicionada pelo SciSpaCy")

    # -------------------------------
    # --- Distribuição por tipo com votação ---
    # -------------------------------
    entidades = {"DOENÇA": set(), "SINTOMA": set(), "TRATAMENTO": set()}

    priority_map = {
        "Sign_symptom": "SINTOMA",
        "Symptom": "SINTOMA",
        "SIGN": "SINTOMA",
        "PROBLEM": "SINTOMA",
        "TREATMENT": "TRATAMENTO",
        "PROCEDURE": "TRATAMENTO",
        "DISEASE": "DOENÇA",
        "Therapeutic_procedure": "TRATAMENTO",
    }

    print("\n[DEBUG] --- Distribuição por tipo com votos e prioridade ---")
    for entidade, labels in entity_votes.items():
        mapped_labels = [priority_map.get(l) for l in labels if l in priority_map]
        if not mapped_labels:
            continue

        if entidade in sintomas_comuns:
            final_label = "SINTOMA"
        elif "SINTOMA" in mapped_labels:
            final_label = "SINTOMA"
        elif "TRATAMENTO" in mapped_labels:
            final_label = "TRATAMENTO"
        else:
            final_label = "DOENÇA"

        entidades[final_label].add(entidade)
        print(f"Entidade: {entidade}, Labels associadas: {labels}")
        print(f"  → Voto final: {final_label} para {entidade}")

    # -------------------------------
    # --- DrugBank reforça tratamentos ---
    # -------------------------------
    print("\n[DEBUG] --- DrugBank ---")
    for drug in drugbank_list:
        if re.search(r'\b' + re.escape(drug) + r'\b', texto_lower):
            entidades["TRATAMENTO"].add(drug)
            print(f"  → DRUGBANK TRATAMENTO: {drug}")

    # -------------------------------
    # --- Heurística de Conflito de Substring ---
    # -------------------------------
    print("\n[DEBUG] --- Corrigindo conflitos de substring (ex: Doença vs Tratamento) ---")
    doencas_a_remover = set()
    tratamentos_a_adicionar = set()

    for doenca in entidades["DOENÇA"]:
        for tratamento in entidades["TRATAMENTO"]:
            if tratamento in doenca:
                print(f"  → Conflito: DOENÇA '{doenca}' contém TRATAMENTO '{tratamento}'.")
                print(f"     → Movendo '{doenca}' para TRATAMENTO.")
                doencas_a_remover.add(doenca)
                tratamentos_a_adicionar.add(doenca)
                break 

    entidades["DOENÇA"].difference_update(doencas_a_remover)
    entidades["TRATAMENTO"].update(tratamentos_a_adicionar)

    # -------------------------------
    # --- Limpeza de Ruído (Filtro) ---
    # -------------------------------
    print("\n[DEBUG] --- Filtrando ruído (ex: <= 2 caracteres) ---")
    excecoes_curtas = {drug for drug in drugbank_list if len(drug) <= 2}
    for k in entidades:
        entidades_a_remover = set()
        for entidade in entidades[k]:
            if len(entidade) <= 2 and entidade not in excecoes_curtas:
                entidades_a_remover.add(entidade)
                print(f"  → Removendo ruído: {entidade} (Categoria: {k})")
        entidades[k] -= entidades_a_remover
        
    # -------------------------------
    # --- AJUSTE GENÉRICO DE SINTOMAS ---
    # -------------------------------
    print("\n[DEBUG] --- Removendo sintomas genéricos (ex: symptoms) ---")
    termos_genericos = {"symptoms", "signs", "issues", "problems", "complaints"}
    for termo in termos_genericos:
        if termo in entidades["SINTOMA"]:
            print(f"  → Removendo termo genérico: {termo}")
            entidades["SINTOMA"].discard(termo)
    
    # -------------------------------
    # --- HEURÍSTICA: Fundir Entidades Adjacentes ---
    # -------------------------------
    print("\n[DEBUG] --- Tentando fundir entidades adjacentes (ex: fetal hydrops) ---")
    for k in entidades:
        print(f"  → Verificando fusões em: {k}")
        entidades[k] = fundir_entidades_adjacentes(entidades[k], texto_lower)

    # -------------------------------
    # --- Limpeza final ---
    # -------------------------------
    print("\n[DEBUG] --- Limpeza final (unindo substrings) ---")
    for k in entidades:
        entidades[k] = unir_substrings(entidades[k])

    return {k: sorted(list(v)) for k, v in entidades.items()}

# --- Aplicar aos artigos ---
def entities_from_title_abstract(row):
    # Assegura que title e abstract não são nulos
    titulo = str(row.get('title', ''))
    abstract = str(row.get('abstract', ''))
    
    texto = f"{titulo} {abstract}"
    return extrair_entidades(str(texto))

    
df = pd.read_csv("articles.csv") 

if 'df' in locals():
    print(f"Processando {len(df)} artigos...")
    df["entities"] = df.apply(entities_from_title_abstract, axis=1)

    # --- Mostrar alguns resultados corrigido ---
    entities_data = []
    for i, row in df.iterrows():
        entities_data.append({
           "pmid": row["pmid"],
           "DOENÇA": ", ".join(row["entities"]["DOENÇA"]),
           "SINTOMA": ", ".join(row["entities"]["SINTOMA"]),
           "TRATAMENTO": ", ".join(row["entities"]["TRATAMENTO"])
        })

    entities_df = pd.DataFrame(entities_data)
    entities_df.to_csv("entities.csv", index=False)
    print("\n\n--- RESULTADO FINAL (entities.csv) ---")

    for i, row in entities_df.iterrows():
        print(f"\nPMID: {row['pmid']}")
        print(f"  DOENÇA: {row['DOENÇA'] or 'Nenhuma'}")
        print(f"  SINTOMA: {row['SINTOMA'] or 'Nenhuma'}")
        print(f"  TRATAMENTO: {row['TRATAMENTO'] or 'Nenhuma'}")

Device set to use cpu


AttributeError: 'NoneType' object has no attribute 'strip'

## *Extração de Relações*

In [43]:
# ===============================================
# ====== EXTRAÇÃO DE RELAÇÕES (ABORDAGEM ZERO-SHOT) =
# ===============================================
import spacy
from transformers import pipeline
import torch
import itertools
import pandas as pd # Certifique-se que o pandas está importado

# --- 1. Carregar Modelo de Sentenças (igual ao que tinha) ---
nlp_sent = spacy.blank("en")
nlp_sent.add_pipe("sentencizer")

# --- 2. Carregar o Pipeline de Classificação "Zero-Shot" ---
print("Carregando modelo Zero-Shot (pode demorar)...")

# NOTA: Este é um modelo geral. Para melhores resultados, pode trocá-lo
# por um modelo NLI focado em biomedicina, como:
# "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext-finetuned-mnli"
# Mas vamos usar o 'bart-large-mnli' que é o padrão e funciona bem.
re_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1 # Usar GPU se disponível
)

# --- 3. DEFINIR AS SUAS PRÓPRIAS RELAÇÕES ---
# Estas são as "hipóteses" que o modelo vai testar.
# Pode alterar/adicionar as que quiser!
CUSTOM_RELATION_LABELS = [
    "treats",             # Trata
    "causes",             # Causa
    "is a symptom of",    # É um sintoma de
    "is a side effect of",# É um efeito colateral de
    "is not related"      # Não está relacionado (importante!)
]

# --- 4. Função para classificar relação (MODIFICADA) ---
def classify_relation_zero_shot(e1, e2, sentence):
    """
    Usa o pipeline "zero-shot" para classificar a relação
    entre duas entidades numa frase.
    """
    
    # Criar "hipóteses" dinâmicas para o modelo testar
    # Ex: "ibuprofen treats headache"
    # Ex: "ibuprofen causes headache"
    # ...
    hypotheses = [f"{e1} {label} {e2}" for label in CUSTOM_RELATION_LABELS]
    
    try:
        # A premissa é a frase inteira
        premise = sentence
        
        # O pipeline testa a 'premise' contra todas as 'hypotheses'
        result = re_classifier(premise, hypotheses, multi_label=False)
        
        # O resultado é um dicionário, ex:
        # {'sequence': '...', 
        #  'labels': ['treats', 'is not related', 'causes', ...], 
        #  'scores': [0.95, 0.02, 0.01, ...]}
        
        best_label = result["labels"][0]
        best_score = result["scores"][0]
        
        # DEFINIR UM LIMIAR DE CONFIANÇA
        # Se a melhor relação for "não relacionado", ou se a pontuação
        # for muito baixa, ignoramos.
        if best_label == "is not related" or best_score < 0.7:
            return None
            
        return best_label

    except Exception as e:
        print(f"Erro no pipeline de classificação: {e}")
        return None

# --- 5. Função principal de relação (QUASE IGUAL À SUA) ---
def extrair_relacoes(texto, entidades):
    # 'entidades' deve ser o dicionário que você já tem, ex:
    # {'TRATAMENTO': ['aspirin'], 'DOENÇA': ['fever']}
    
    relacoes = []
    doc_sent = nlp_sent(texto)

    for sent in doc_sent.sents:
        sent_text = sent.text.lower() # Converter para minúsculas

        # Encontrar entidades PRESENTES nesta frase específica
        ents_in_sent = {}
        for tipo, lista in entidades.items():
            ents_in_sent[tipo] = [e.lower() for e in lista if e.lower() in sent_text]

        # Se não houver pelo menos duas entidades na frase, saltar
        total_ents_in_sent = sum(len(v) for v in ents_in_sent.values())
        if total_ents_in_sent < 2:
            continue

        # Gerar todos os pares possíveis (Tratamento-Doença, etc.)
        pares = []
        
        # (Tratamento ↔ Doença)
        pares += list(itertools.product(ents_in_sent.get("TRATAMENTO", []), ents_in_sent.get("DOENÇA", [])))
        # (Tratamento ↔ Sintoma)
        pares += list(itertools.product(ents_in_sent.get("TRATAMENTO", []), ents_in_sent.get("SINTOMA", [])))
        # (Sintoma ↔ Doença)
        pares += list(itertools.product(ents_in_sent.get("SINTOMA", []), ents_in_sent.get("DOENÇA", [])))
        # (Tratamento ↔ Tratamento) - O seu DDI original
        pares += list(itertools.product(ents_in_sent.get("TRATAMENTO", []), ents_in_sent.get("TRATAMENTO", [])))
        
        # Classificar cada par
        for e1, e2 in pares:
            if e1 == e2: continue # Evitar pares da mesma entidade
                
            # Usar a nova função de classificação
            rel = classify_relation_zero_shot(e1, e2, sent.text)
            
            # Se uma relação válida (e acima do limiar) foi encontrada
            if rel:
                relacoes.append({
                    "Entity1": e1,
                    "Entity2": e2,
                    "Relation": rel,
                    "Sentence": sent.text
                })

    return relacoes

# --- 6. Aplicar ao seu DataFrame (ASSUMINDO QUE 'df' EXISTE) ---

# Criar um DataFrame de exemplo caso estejamos a correr
# o script de forma isolada
try:
    df
except NameError:
    print("DataFrame 'df' não encontrado. Criando um exemplo...")
    df_data = {
        'title': [
            "Study finds Aspirin is effective for fever and headache",
            "Metformin side effects include nausea"
        ],
        'abstract': [
            "We tested aspirin against common cold symptoms. The primary symptom was fever.",
            "A common side effect of metformin is nausea. This drug treats diabetes."
        ],
        'entities': [
            {'TRATAMENTO': ['Aspirin'], 'DOENÇA': ['headache', 'common cold'], 'SINTOMA': ['fever']},
            {'TRATAMENTO': ['Metformin'], 'SINTOMA': ['nausea'], 'DOENÇA': ['diabetes']}
        ]
    }
    df = pd.DataFrame(df_data)


# --- Aplicar aos artigos (O SEU CÓDIGO ORIGINAL) ---
print("\nExtraindo relações com o modelo Zero-Shot...")
df["relations"] = df.apply(
    lambda row: extrair_relacoes(str(row["title"]) + " " + str(row["abstract"]), row["entities"]),
    axis=1
)

df.to_csv("articles_with_relations_zero_shot.csv", index=False)
print("🟢 Relações gravadas em 'articles_with_relations_zero_shot.csv'")

# Opcional: Ver os resultados
print("\nResultados:")
for idx, row in df.iterrows():
    print(f"--- Artigo {idx+1} ---")
    if row['relations']:
        for rel in row['relations']:
            print(f"  {rel['Entity1']} --[{rel['Relation']}]--> {rel['Entity2']}")
    else:
        print("  Nenhuma relação encontrada.")

Carregando modelo Zero-Shot (pode demorar)...


c:\Users\ferna\anaconda3\envs\CLProject\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ferna\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular


Extraindo relações com o modelo Zero-Shot...
🟢 Relações gravadas em 'articles_with_relations_zero_shot.csv'

Resultados:
--- Artigo 1 ---
  fetal hydrops --[fetal hydrops is a symptom of ballantyne syndrome]--> ballantyne syndrome
  cesarean section --[cesarean section treats preterm premature rupture]--> preterm premature rupture


In [69]:
# ===============================================
# ====== EXTRAÇÃO DE RELAÇÕES (ENSEMBLE 2/3) =
# ===============================================
import spacy
from transformers import pipeline, AutoTokenizer
import torch
import itertools
import pandas as pd
import warnings
from collections import Counter # Importar o Counter para a votação

# Suprimir avisos
warnings.filterwarnings(
    "ignore", 
    message="Token indices sequence length is longer than the specified maximum sequence length.*"
)

# --- 1. Carregar Modelo de Sentenças ---
nlp_sent = spacy.blank("en")
nlp_sent.add_pipe("sentencizer")

# --- 2. Carregar o Pipeline de Classificação (★ 3 MODELOS ★) ---

# --- MODELO 1: Generalista (BART) ---
print("Carregando Modelo 1 (Generalista - BART)...")
re_classifier_bart = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1 
)

# --- MODELO 2: Especialista (PubMedBERT) ---
print("Carregando Modelo 2 (Especialista - PubMedBERT)...")
MODEL_NAME_BERT = "pritamdeka/PubMedBERT-MNLI-MedNLI"
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME_BERT)
re_classifier_bert = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME_BERT,
    tokenizer=tokenizer_bert,
    device=0 if torch.cuda.is_available() else -1
)

# --- MODELO 3: Especialista NLI (DeBERTa) (★ NOVO MODELO ★) ---
print("Carregando Modelo 3 (Especialista NLI - DeBERTa)...")
MODEL_NAME_DEBERTA = "cross-encoder/nli-deberta-v3-base"
tokenizer_deberta = AutoTokenizer.from_pretrained(MODEL_NAME_DEBERTA)
re_classifier_deberta = pipeline(
    "zero-shot-classification",
    model=MODEL_NAME_DEBERTA,
    tokenizer=tokenizer_deberta,
    device=0 if torch.cuda.is_available() else -1
)

# --- 3. Definir Relações ---
CUSTOM_RELATION_LABELS = [
    "treats",
    "causes",
    "is a symptom of",
    "is a side effect of",
    "is not related"
]

# --- 4. Função de Classificação (Genérica - Sem Alterações) ---
def classify_relation_with_model(e1, e2, sentence, classifier, confidence_threshold):
    """
    Função genérica que classifica uma relação usando um pipeline
    e um limiar de confiança específicos.
    """
    hypotheses = [f"{e1} {label} {e2}" for label in CUSTOM_RELATION_LABELS]
    
    try:
        max_len = classifier.tokenizer.model_max_length
        
        tokenized_sentence = classifier.tokenizer(sentence, truncation=False)["input_ids"]
        if len(tokenized_sentence) > max_len:
            truncated_ids = tokenized_sentence[:max_len-1] + [tokenized_sentence[-1]]
            premise = classifier.tokenizer.decode(truncated_ids, skip_special_tokens=True)
        else:
            premise = sentence
        
        result = classifier(premise, hypotheses, multi_label=False)
        
        best_label = result["labels"][0]
        best_score = result["scores"][0]
        
        if best_label == "is not related" or best_score < confidence_threshold:
            return None, best_score # Retorna None E o score (para debug)
            
        return best_label, best_score

    except Exception as e:
        print(f"Erro no pipeline de classificação: {e}\nFrase: {sentence}\n")
        return None, 0.0

# --- 5. Função Principal (★ LÓGICA DE VOTAÇÃO REVISTA ★) ---
def extrair_relacoes(texto, entidades, conf_thresh_bart, conf_thresh_bert, conf_thresh_deberta):
    
    relacoes = []
    
    if not isinstance(entidades, dict):
        return [] 
        
    entidades_padronizadas = {}
    for tipo, lista in entidades.items():
        if isinstance(lista, (list, set)):
            entidades_padronizadas[tipo] = [str(e).lower().strip() for e in lista]
        
    doc_sent = nlp_sent(texto)

    for sent in doc_sent.sents:
        sent_text_lower = sent.text.lower()

        ents_in_sent = {}
        for tipo, lista in entidades_padronizadas.items():
            ents_in_sent[tipo] = [e for e in lista if e in sent_text_lower]

        if sum(len(v) for v in ents_in_sent.values()) < 2:
            continue

        pares = []
        pares += list(itertools.product(ents_in_sent.get("TRATAMENTO", []), ents_in_sent.get("DOENÇA", [])))
        pares += list(itertools.product(ents_in_sent.get("TRATAMENTO", []), ents_in_sent.get("SINTOMA", [])))
        pares += list(itertools.product(ents_in_sent.get("SINTOMA", []), ents_in_sent.get("DOENÇA", [])))
        pares += list(itertools.product(ents_in_sent.get("TRATAMENTO", []), ents_in_sent.get("TRATAMENTO", [])))
        
        for e1, e2 in pares:
            if e1 == e2: continue
                
            # --- Classificar com os 3 Modelos ---
            rel_bart, score_bart = classify_relation_with_model(
                e1, e2, sent.text, re_classifier_bart, conf_thresh_bart
            )
            rel_bert, score_bert = classify_relation_with_model(
                e1, e2, sent.text, re_classifier_bert, conf_thresh_bert
            )
            rel_deberta, score_deberta = classify_relation_with_model(
                e1, e2, sent.text, re_classifier_deberta, conf_thresh_deberta
            )
            
            # --- DEBUG ---
            print(f"\n--- DEBUGGING ---")
            print(f"  Frase: {sent.text[:70]}...")
            print(f"  Par: {e1} <--> {e2}")
            print(f"  Modelo BART (Limiar: {conf_thresh_bart}):")
            print(f"    -> Voto: {rel_bart} (Score: {score_bart:.4f})")
            print(f"  Modelo BERT (Limiar: {conf_thresh_bert}):")
            print(f"    -> Voto: {rel_bert} (Score: {score_bert:.4f})")
            print(f"  Modelo DeBERTa (Limiar: {conf_thresh_deberta}):")
            print(f"    -> Voto: {rel_deberta} (Score: {score_deberta:.4f})")
            
            # --- ★ LÓGICA DE VOTAÇÃO (MAIORIA 2/3) ★ ---
            votos_validos = [v for v in [rel_bart, rel_bert, rel_deberta] if v is not None]
            
            if len(votos_validos) < 2: # Precisa de pelo menos 2 votos para ter maioria
                print("  ==> VOTAÇÃO REJEITADA. (Menos de 2 modelos atingiram o limiar)")
                continue

            # Encontrar o voto mais comum
            contagem_votos = Counter(votos_validos)
            rel_final, num_votos = contagem_votos.most_common(1)[0]
            
            if num_votos >= 2:
                print(f"  ==> VOTAÇÃO APROVADA! ({num_votos}/3 concordam em '{rel_final}')")
                
                # Coletar scores para o log (se o modelo votou nesta relação)
                score_g = score_bart if rel_bart == rel_final else 0.0
                score_b = score_bert if rel_bert == rel_final else 0.0
                score_d = score_deberta if rel_deberta == rel_final else 0.0
                
                relacoes.append({
                    "Entity1": e1,
                    "Entity2": e2,
                    "Relation": rel_final,
                    "Score_BART": score_g,
                    "Score_BERT": score_b,
                    "Score_DeBERTa": score_d,
                    "Sentence": sent.text
                })
            else:
                print(f"  ==> VOTAÇÃO REJEITADA. (Sem maioria. Votos: {dict(contagem_votos)})")

    return relacoes

# --- 6. Aplicar ao DataFrame (★ 3 LIMIARES ★) ---
try:
    df
except NameError:
    print("DataFrame 'df' não encontrado. Criando um exemplo com as ENTIDADES REAIS...")
    df_data = {
        'title': ["Successful management of parvovirus B19-associated maternal mirror syndrome..."],
        'abstract': [
            "Maternal mirror syndrome (MMS), or Ballantyne syndrome, is a rare complication... characterized by... fetal hydrops. ...PVB19 infection. ...diagnosed with severe fetal anemia... and underwent intrauterine blood transfusions (IBT). ...Treatment with albumin and furosemide effectively stabilized her condition... significant anasarca... Delivery occurred via cesarean section... following preterm premature rupture of membranes..."
        ],
        # ★ ESTA É A LISTA REAL DO SEU NER SCRIPT ★
        'entities': [
            {
                'DOENÇA': ['anasarca', 'ballantyne syndrome', 'preterm premature rupture', 'pvb19 infection'],
                'SINTOMA': ['anemia', 'fetal hydrops'],
                'TRATAMENTO': ['cesarean section', 'furosemide', 'intrauterine blood transfusions']
            }
        ]
    }
    df = pd.DataFrame(df_data)


# ★ MUDANÇA 2: DEFINIR OS LIMIARES RELAXADOS ★
# O nosso objetivo é apanhar mais relações (baixar falsos negativos)
LIMIAR_BART = 0
LIMIAR_BERT = 0
LIMIAR_DEBERTA = 0

print(f"\nExtraindo relações com ENSEMBLE 2/3 (Limiares RELAXADOS: BART={LIMIAR_BART}, BERT={LIMIAR_BERT}, DeBERTa={LIMIAR_DEBERTA})...")
df["relations"] = df.apply(
    lambda row: extrair_relacoes(
        str(row["title"]) + " " + str(row["abstract"]), 
        row["entities"],
        conf_thresh_bart=LIMIAR_BART,
        conf_thresh_bert=LIMIAR_BERT,
        conf_thresh_deberta=LIMIAR_DEBERTA
    ),
    axis=1
)

df.to_csv("articles_with_relations_ENSEMBLE_v5_relaxed.csv", index=False)
print("🟢 Relações gravadas em 'articles_with_relations_ENSEMBLE_v5_relaxed.csv'")

# --- Ver os resultados ---
print("\nResultados Finais:")
for idx, row in df.iterrows():
    print(f"--- Artigo {idx+1} ---")
    if row['relations']:
        for rel in row['relations']:
            print(f"  {rel['Entity1']} --[{rel['Relation']} (BART: {rel['Score_BART']:.2f}, BERT: {rel['Score_BERT']:.2f}, DeBERTa: {rel['Score_DeBERTa']:.2f})]--> {rel['Entity2']}")
    else:
        print("  Nenhuma relação encontrada.")

Carregando Modelo 1 (Generalista - BART)...


Device set to use cpu


Carregando Modelo 2 (Especialista - PubMedBERT)...


Device set to use cpu


Carregando Modelo 3 (Especialista NLI - DeBERTa)...


Device set to use cpu



Extraindo relações com ENSEMBLE 2/3 (Limiares RELAXADOS: BART=0, BERT=0, DeBERTa=0)...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



--- DEBUGGING ---
  Frase: Maternal mirror syndrome (MMS), or Ballantyne syndrome, is a rare comp...
  Par: fetal hydrops <--> ballantyne syndrome
  Modelo BART (Limiar: 0):
    -> Voto: fetal hydrops is a symptom of ballantyne syndrome (Score: 0.7272)
  Modelo BERT (Limiar: 0):
    -> Voto: fetal hydrops is a symptom of ballantyne syndrome (Score: 0.4470)
  Modelo DeBERTa (Limiar: 0):
    -> Voto: fetal hydrops is a side effect of ballantyne syndrome (Score: 0.6342)
  ==> VOTAÇÃO APROVADA! (2/3 concordam em 'fetal hydrops is a symptom of ballantyne syndrome')

--- DEBUGGING ---
  Frase: A 38-year-old G2P1 patient at 27 + 2 weeks was diagnosed with severe f...
  Par: intrauterine blood transfusions <--> anemia
  Modelo BART (Limiar: 0):
    -> Voto: intrauterine blood transfusions treats anemia (Score: 0.6420)
  Modelo BERT (Limiar: 0):
    -> Voto: intrauterine blood transfusions is a symptom of anemia (Score: 0.3693)
  Modelo DeBERTa (Limiar: 0):
    -> Voto: intrauterine blood tran